# CALVIN 数据集探索

使用 LeRobot 内置的 `LeRobotDataset` 加载 CALVIN 数据，
可视化样本帧与动作分布，验证数据加载管线。

数据来源: HuggingFace Hub `lerobot/calvin` (LeRobot 格式)

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import torch
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
# 从 HuggingFace Hub 加载 CALVIN 数据集
from lerobot.datasets.lerobot_dataset import LeRobotDataset

try:
    dataset = LeRobotDataset("lerobot/calvin", split="train")
    print(f"数据集加载成功!")
    print(f"总帧数: {len(dataset)}")
    print(f"Episodes: {dataset.num_episodes}")
    print(f"Camera 名称: {dataset.camera_names}")
    print(f"特征: {list(dataset.features.keys())}")
except Exception as e:
    print(f"从 Hub 加载失败: {e}")
    print("尝试从本地加载...")
    # 回退: 从本地 LeRobot 格式目录加载
    dataset = LeRobotDataset("./data/calvin_lerobot", split="train")

In [ ]:
# 查看一个样本
sample = dataset[0]
print("样本键:", list(sample.keys()))
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# 可视化前 10 帧 (两个相机视角)
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i in range(5):
    obs = dataset[i]
    # LeRobot 图像格式: (C, H, W) float32 [0,1]
    static = obs.get("observation.images.rgb_static", obs.get("rgb_static"))
    gripper = obs.get("observation.images.rgb_gripper", obs.get("rgb_gripper"))
    
    if static.ndim == 3:
        static_img = static.permute(1, 2, 0).numpy()
    else:
        static_img = static.numpy()
    if gripper.ndim == 3:
        gripper_img = gripper.permute(1, 2, 0).numpy()
    else:
        gripper_img = gripper.numpy()
    
    axes[0, i].imshow(static_img)
    axes[0, i].set_title(f"Static Frame {i}")
    axes[0, i].axis('off')
    axes[1, i].imshow(gripper_img)
    axes[1, i].set_title(f"Gripper Frame {i}")
    axes[1, i].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 动作分布可视化
all_actions = []
for i in range(0, min(500, len(dataset)), 5):
    obs = dataset[i]
    action = obs.get("action", obs.get("actions"))
    if isinstance(action, torch.Tensor):
        all_actions.append(action.numpy())

all_actions = np.stack(all_actions, axis=0)
print(f"动作数据形状: {all_actions.shape}")  # (N, chunk_size, action_dim)

action_labels = ['dx', 'dy', 'dz', 'droll', 'dpitch', 'dyaw', 'gripper']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(7):
    ax = axes[i // 4, i % 4]
    ax.hist(all_actions[:, 0, i].flatten(), bins=50, alpha=0.7, color='#2196F3')
    ax.set_title(action_labels[i])
    ax.set_xlabel('Value')
axes[1, 3].axis('off')
plt.suptitle('动作分布 (chunk 第一步)')
plt.tight_layout()
plt.show()

In [ ]:
# 数据集统计信息
n_episodes = dataset.num_episodes
n_frames = len(dataset)
print(f"数据集统计:")
print(f"  总 Episodes: {n_episodes}")
print(f"  总帧数: {n_frames}")
print(f"  平均帧/episode: {n_frames / n_episodes:.1f}")

# 动作统计
flat_actions = all_actions[:, 0, :].reshape(-1, 7)
for i, label in enumerate(action_labels):
    print(f"  {label:8s}: mean={flat_actions[:, i].mean():+.4f}, std={flat_actions[:, i].std():.4f}")